In [ ]:
import pandas as pd
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../data/processed/training_data_with_history.csv")
df["date"] = pd.to_datetime(df["date"])

with open("../data/processed/selected_features.json", "r") as file:
    raw_features = json.load(file)

history_features = ["smart_5_raw_change_1d", "smart_5_raw_change_7d", "smart_5_raw_change_30d", "smart_5_raw_rolling_mean_7d", "smart_5_raw_rolling_std_7d", "smart_5_raw_rolling_mean_30d"]

raw_and_history_features = raw_features + history_features

df = df.sort_values("date").reset_index(drop=True)

print(df)

Raw features: 27
Raw + history features: 33
       serial_number       date                 model  capacity_bytes  \
0           1RJEXUVG 2026-01-01   WDC WUH722222ALE6L4  22000969973760   
1       11J0A0JGF97G 2026-01-01   TOSHIBA MG07ACA14TA  14000519643136   
2           ZL2351Q6 2026-01-01         ST16000NM001G  16000900661248   
3       6240A26FFVKG 2026-01-01   TOSHIBA MG08ACA16TA  16000900661248   
4           ZL2MM80H 2026-01-01         ST16000NM001G  16000900661248   
...              ...        ...                   ...             ...   
102118  74N0A0PXF4MJ 2026-03-30   TOSHIBA MG10ACA20TE  20000588955648   
102119      2BJB42BN 2026-03-30   WDC WUH721816ALE6L4  16000900661248   
102120      ZA145EFE 2026-03-30          ST8000NM0055   8001563222016   
102121  41B0A029FV8G 2026-03-30  TOSHIBA MG08ACA16TEY  16000900661248   
102122      ZL22YS77 2026-03-30         ST16000NM001G  16000900661248   

        failure datacenter  cluster_id  vault_id  pod_id  pod_slot_num  ...  \


In [ ]:
raw_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("model", RandomForestClassifier(random_state=44))]) 
history_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("model", RandomForestClassifier(random_state=44))]) 

X_raw = df[raw_features]
X_history = df[raw_and_history_features]

y = df["failure_within_30_days"]

train_end = pd.Timestamp("2026-02-09")
val_end = pd.Timestamp("2026-02-19")

train_mask = df["date"] <= train_end
val_mask = ((df["date"] > train_end) & (df["date"] <= val_end))

X_train_raw = X_raw.loc[train_mask]
X_val_raw = X_raw.loc[val_mask]

X_train_history = X_history.loc[train_mask]
X_val_history = X_history.loc[val_mask]

y_train = y.loc[train_mask]
y_val = y.loc[val_mask]

raw_pipeline.fit(X_train_raw, y_train)
history_pipeline.fit(X_train_history, y_train)